# ME-model Build

Combines a morphology and an e-model into an ME-model, then calibrates and validates it. See `docs/memodel-build.md`.

In [ ]:
from entitysdk import Client, ProjectContext
from obi_auth import get_token

from obi_one import LAB_ID_STAGING_TEST, PROJECT_ID_STAGING_TEST
from obi_one.core.info import Info
from obi_one.core.run_tasks import run_tasks_for_generated_scan
from obi_one.core.scan_generation import GridScanGenerationTask
from obi_one.scientific.from_id.cell_morphology_from_id import CellMorphologyFromID
from obi_one.scientific.from_id.emodel_from_id import EModelFromID
from obi_one.scientific.tasks.memodel_build.config import MEModelBuildScanConfig

obi_one_api_url = "http://127.0.0.1:8100"

token = get_token(environment="staging")
project_context = ProjectContext(virtual_lab_id=LAB_ID_STAGING_TEST, project_id=PROJECT_ID_STAGING_TEST)
db_client = Client(
    api_url="https://staging.openbraininstitute.org/api/entitycore",
    project_context=project_context,
    token_manager=token,
)

# Morphology and e-model

One of each. Check them first with the Bluenaas compatibility check (`POST /single-neuron/compatibility/run`), as the platform does.

In [ ]:
morphology_id = "<cell morphology id>"
emodel_id = "<e-model id>"

# Config

In [ ]:
config = MEModelBuildScanConfig(
    info=Info(campaign_name="ME-model build example", campaign_description="Built from the example notebook"),
    initialize=MEModelBuildScanConfig.Initialize(
        morphology=CellMorphologyFromID(id_str=morphology_id),
        emodel=EModelFromID(id_str=emodel_id),
    ),
)

# Generate and build locally

Registers the campaign, the config and the ME-model, then calibrates and validates it. Needs NEURON (`nrnivmodl`) on the `PATH`.

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as temp_dir:
    grid_scan = GridScanGenerationTask(form=config, coordinate_directory_option="ZERO_INDEX", output_root=temp_dir)
    grid_scan.execute(db_client=db_client)
    run_tasks_for_generated_scan(grid_scan, db_client=db_client)

# Generate through the service

The service must be running (`make run-local`). It registers the campaign and the config without building; Bluenaas or the launch system runs the build from the config id.

In [ ]:
import requests

response = requests.post(
    f"{obi_one_api_url}/generated/me-model-build-scan-config-generate-grid",
    headers={
        "Authorization": f"Bearer {token}",
        "virtual-lab-id": LAB_ID_STAGING_TEST,
        "project-id": PROJECT_ID_STAGING_TEST,
    },
    json=config.model_dump(mode="json"),
    timeout=60,
)
response.raise_for_status()
campaign_id = response.json()
campaign_id